# DistilBERT — MayI tool-call classifier

`Runtime > Change runtime type > T4 GPU` seç, sonra *Run all*.

Notebook kodu kendi içinde taşımaz: takım reposunu indirip `distilbert/` klasöründeki script'leri çalıştırır. Çıktılar Google Drive'a (`MyDrive/mayi_distilbert/`) yazılır; Colab koparsa aynı hücreyi tekrar çalıştırınca son checkpoint'ten devam eder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/mayi_distilbert'
!mkdir -p $OUT

In [ ]:
# Code repo (holds train_distilbert.py and evaluate_predictions.py)
REPO = 'https://github.com/akifkulll/mayi-distilbert'

import os
%cd /content
!test -d project || git clone -q $REPO project
!test -d LucidAI-Dataset || git clone -q https://github.com/Enes-Ozbek/LucidAI-Dataset
# scripts sit at the repo root, or under distilbert/ once merged into the team repo
os.chdir('/content/project/distilbert' if os.path.isdir('/content/project/distilbert') else '/content/project')
DATA = '/content/LucidAI-Dataset/data'
!ls $DATA

## 1) Smoke test (~2 dk)
Her şeyin çalıştığını küçük veriyle kontrol eder. Hata yoksa devam et.

In [ ]:
!python train_distilbert.py --data-dir $DATA --input a --smoke --out /content/smoke --predict val

## 2) Girdi A: sadece komut (~100k satır, 128 token)

In [ ]:
!python train_distilbert.py --data-dir $DATA --input a --out $OUT/distilbert_a

## 2b) Girdi A, 60k satır
B ile aynı veri miktarı: A/B karşılaştırmasında tek fark girdi olsun diye.

In [ ]:
!python train_distilbert.py --data-dir $DATA --input a --train-rows 60000 --out $OUT/distilbert_a60k

## 3) Girdi B: komut + istek + geçmiş (~60k satır, 512 token)
En uzun adım bu. Koparsa bu hücreyi tekrar çalıştır, kaldığı yerden devam eder.

In [ ]:
!python train_distilbert.py --data-dir $DATA --input b --out $OUT/distilbert_b

## 4) Puanlama

In [ ]:
for run in ['distilbert_a', 'distilbert_a60k', 'distilbert_b']:
    for split in ['val', 'test', 'ood_approve_or_deny', 'ood_saroku']:
        !python evaluate_predictions.py --pred $OUT/$run/predictions/{split}.csv --data-dir $DATA --split {split} --out $OUT/results/{run}_{split}

In [ ]:
import json, pandas as pd
rows = []
for run in ['distilbert_a', 'distilbert_a60k', 'distilbert_b']:
    for split in ['test', 'ood_approve_or_deny', 'ood_saroku']:
        m = json.load(open(f'{OUT}/results/{run}_{split}/metrics.json'))['overall']
        rows.append({'run': run, 'split': split, **{k: round(m[k], 4) for k in ['accuracy', 'macro_f1', 'false_allow_rate', 'false_deny_rate', 'auroc']}})
pd.DataFrame(rows)

In [ ]:
print(open(f'{OUT}/results/distilbert_b_test/metrics.md').read())